In [ ]:
import rasterio
import numpy as np
import os

# --------------------------------------------------------------------
# User inputs
# --------------------------------------------------------------------
rcmap_path = "RCMAP30m_23.tif"
rap_path   = "RAP_2023.tif"
out_diff_path = "RCMAP_RAP_diff.tif"

# Mapping: {RCMAP_band: RAP_band}
band_map = {
    1: 5,  # Shrub
    2: 1,  # Annual herbaceous / Annual forbs & grass
    4: 3,  # Litter
    6: 2,  # Bareground
    7: 4   # Perennial herbaceous / Perennial forbs & grass
}

# Nodata value for output
OUT_NODATA = -9999.0

# --------------------------------------------------------------------
# Open rasters
# --------------------------------------------------------------------
with rasterio.open(rcmap_path) as src_rcmap, rasterio.open(rap_path) as src_rap:

    # Basic sanity checks
    if src_rcmap.crs != src_rap.crs:
        raise ValueError(f"CRS mismatch: RCMAP={src_rcmap.crs}, RAP={src_rap.crs}")

    if src_rcmap.transform != src_rap.transform:
        raise ValueError("Spatial transform mismatch between RCMAP and RAP. "
                         "Resample/reproject one to match the other first.")

    if (src_rcmap.width, src_rcmap.height) != (src_rap.width, src_rap.height):
        raise ValueError("Dimension mismatch between RCMAP and RAP. "
                         "Resample/reproject one to match the other first.")

    print("Rasters align in CRS, transform, and shape. Proceeding...")

    rc_nodata = src_rcmap.nodata
    rap_nodata = src_rap.nodata

    diff_arrays = []
    band_names = []

    # ----------------------------------------------------------------
    # Loop over mapped bands and compute differences
    # ----------------------------------------------------------------
    for rc_band, rap_band in band_map.items():
        print(f"\nProcessing RCMAP band {rc_band} vs RAP band {rap_band}...")

        rc = src_rcmap.read(rc_band).astype("float32")
        rap = src_rap.read(rap_band).astype("float32")

        # Build valid mask (exclude nodata)
        valid_mask = np.ones(rc.shape, dtype=bool)

        if rc_nodata is not None:
            valid_mask &= (rc != rc_nodata)
        if rap_nodata is not None:
            valid_mask &= (rap != rap_nodata)

        # Initialize diff with nodata
        diff = np.full(rc.shape, np.nan, dtype="float32")

        # Compute difference where valid (RCMAP - RAP)
        diff[valid_mask] = rc[valid_mask] - rap[valid_mask]

        # Store for writing later
        diff_arrays.append(diff)

        # Some basic statistics
        abs_diff = np.abs(diff[valid_mask])
        mae = np.nanmean(abs_diff)
        rmse = np.sqrt(np.nanmean(diff[valid_mask] ** 2))
        mean_diff = np.nanmean(diff[valid_mask])

        print(f"  Mean difference (RCMAP - RAP): {mean_diff:.3f}")
        print(f"  MAE (|diff|): {mae:.3f}")
        print(f"  RMSE: {rmse:.3f}")

        band_names.append(f"RC{rc_band}_minus_RAP{rap_band}")

    # ----------------------------------------------------------------
    # Stack differences and write output raster
    # ----------------------------------------------------------------
    diff_stack = np.stack(diff_arrays, axis=0)  # shape = (nbands, rows, cols)

    # Replace NaN with OUT_NODATA
    diff_stack_out = diff_stack.copy()
    diff_stack_out[np.isnan(diff_stack_out)] = OUT_NODATA

    profile = src_rcmap.profile
    profile.update(
        count=len(diff_arrays),
        dtype="float32",
        nodata=OUT_NODATA
    )

    with rasterio.open(out_diff_path, "w", **profile) as dst:
        dst.write(diff_stack_out)

    print(f"\nWrote difference raster to: {os.path.abspath(out_diff_path)}")
    print("Band order in output (RCMAP - RAP):")
    for i, name in enumerate(band_names, start=1):
        print(f"  Band {i}: {name}")
